#### Separando os arquivos de acordo com a coluina Dist Cond AtrioVent


 Primeiro, certifique-se de ter todas as bibliotecas necessárias instaladas
 Se não tiver, descomente as linhas abaixo para instalar
 !pip install pandas
 !pip install openpyxl
 !pip install tqdm
 !pip install psutil


In [1]:
import os
import shutil
import pandas as pd
import re
import logging
from tqdm import tqdm

 Carregando o DataFrame de Chagas a partir do arquivo Excel

In [3]:
#Definindo o caminho para o arquivo Excel
excel_path = '/home/leovitor/Documents/Chagas/Dados/BancoChagas-Geral-Padronizado-25-09-2023.xlsx'

# Carregando o DataFrame a partir do arquivo Excel
df = pd.read_excel(excel_path)
df.head()

,Paciente,Nome do Paciente,filename,Prontuario,Date Nasc,Data Holter,Sexo,Nat,BMI,Cancer,...,Obito,Obito_MS,TendQ - 5º Percentil,TpeakQ - 5º Percentil,% - Qtend/TendQ > 1,% - QTpeak/TpeakQ > 1,98% - Qtend/TendQ,98% - QTpeak/TpeakQ,Tp-Te (ms) - 5º Percentil,Observações
0,NaN,Adão Severo de Souza,1.0,333104.0,1950-11-29,2010-07-08 00:00:00,1.0,PI,25.0,0.0,...,0.0,0.0,0.26,0.33,4.69,1.06,1.79,0.95,46.9,Psoriase
1,NaN,Adão Severo de Souza,2.0,333104.0,1950-11-29,2010-01-14 00:00:00,1.0,PI,23.0,0.0,...,0.0,0.0,0.51,0.58,3.25,1.02,1.83,0.56,54.7,Psoriase
2,NaN,Adão Severo de Souza,3.0,333104.0,1950-11-29,2009-05-27 00:00:00,1.0,PI,24.0,0.0,...,0.0,0.0,0.34,0.41,3.14,0.02,1.04,0.71,54.7,NaN
3,NaN,Adão Severo de Souza,6.0,333104.0,1950-11-29,1997-11-03 00:00:00,1.0,PI,26.0,0.0,...,0.0,0.0,0.35,0.42,4.11,0.29,1.17,0.76,54.7,NaN
4,NaN,Adão Severo de Souza,5.0,333104.0,1950-11-29,2013-05-21 00:00:00,1.0,PI,25.0,0.0,...,0.0,0.0,0.46,0.53,3.61,0.12,1.91,0.59,62.5,NaN


Definindo o caminho base da pasta contendo os arquivos e a função para formatar o nome do arquivo,isso é necessário pois existe uma diferença na formatação dos nomes na planilha e na pasta contendo os arquivos. 

Ex: planilha --> 1.0 | pasta --> 001.txt


In [4]:
# Define o caminho base da pasta contendo os arquivos
base_path = '/home/leovitor/Documents/Chagas/Dados/Exames_24hrs_Recortados'

# Função para formatar o nome do arquivo com base no filename
def format_filename(filename):
    if not pd.isna(filename):  # Verifica se não é NaN
        return f"{int(filename):03d}.txt"
    return None

Função para criar subpastas e mover arquivos com base em Dist Cond AtrioVent

In [6]:

def organizar_arquivos_por_dist_cond(df, base_path):
    for _, row in df.iterrows():
        filename = format_filename(row['filename'])
        dist_cond = str(row['Dist Cond AtrioVent ']).strip()  # Garantir que dist_cond seja uma string e remover espaços extras

        if filename is not None:  # Ignorar se filename é None
            # Define o caminho do arquivo e da subpasta de destino
            file_path = os.path.join(base_path, filename)
            dest_folder = os.path.join(base_path, dist_cond)

            # Move o arquivo para a subpasta de destino apenas se existir na pasta base
            if os.path.exists(file_path):
                # Cria a subpasta se não existir
                if not os.path.exists(dest_folder):
                    os.makedirs(dest_folder)
                
                shutil.move(file_path, os.path.join(dest_folder, filename))
            else:
                print(f"Arquivo {filename} não encontrado em {file_path}")

# Chama a função para organizar os arquivos por Dist Cond AtrioVent
organizar_arquivos_por_dist_cond(df, base_path)


Arquivo 162.txt não encontrado em /home/leovitor/Documents/Chagas/Dados/Exames_24hrs_Recortados/162.txt
Arquivo 163.txt não encontrado em /home/leovitor/Documents/Chagas/Dados/Exames_24hrs_Recortados/163.txt
Arquivo 177.txt não encontrado em /home/leovitor/Documents/Chagas/Dados/Exames_24hrs_Recortados/177.txt
Arquivo 188.txt não encontrado em /home/leovitor/Documents/Chagas/Dados/Exames_24hrs_Recortados/188.txt
Arquivo 303.txt não encontrado em /home/leovitor/Documents/Chagas/Dados/Exames_24hrs_Recortados/303.txt


Função para relacionar a coluna prontuário com a filename, para separar os arquivos por paciente.

In [ ]:
def criar_relacao_filename_prontuario():
    # Caminho do arquivo Excel
    arquivo_excel = "/home/leovitor/Documents/Chagas/Dados/BancoChagas-Geral-Padronizado-25-09-2023.xlsx"

    # Ler o arquivo Excel
    planilha = pd.read_excel(arquivo_excel)

    # Criar um dicionário para armazenar a relação filename-prontuário
    relacao_filename_prontuario = {}

    # Para cada linha na planilha
    for index, row in planilha.iterrows():
        # Verificar se o prontuário é válido
        if pd.notnull(row["Prontuario"]):
            # Verificar se filename é uma lista
            if isinstance(row["filename"], list):
                # Para cada filename na lista de filenames
                for filename in row["filename"]:
                    # Verificar se o filename não é NaN
                    if pd.notnull(filename):
                        try:
                            # Formatando o filename
                            formatted_filename = str(filename).replace('.0', '').zfill(3) if len(str(filename).replace('.0', '')) < 3 else str(filename).replace('.0', '')
                            # Adicionar a relação filename-prontuário ao dicionário
                            relacao_filename_prontuario[formatted_filename] = row["Prontuario"]
                        except Exception as e:
                            print(f"Erro ao processar o arquivo {filename}: {e}")
                            continue
            else:
                # Verificar se o filename não é NaN
                if pd.notnull(row["filename"]):
                    try:
                        # Formatando o filename
                        formatted_filename = str(row["filename"]).replace('.0', '').zfill(3) if len(str(row["filename"]).replace('.0', '')) < 3 else str(row["filename"]).replace('.0', '')
                        # Adicionar a relação filename-prontuário ao dicionário
                        relacao_filename_prontuario[formatted_filename] = row["Prontuario"]
                    except Exception as e:
                        print(f"Erro ao processar o arquivo {row['filename']}: {e}")
                        continue

    return relacao_filename_prontuario

# Criar a relação filename-prontuário
relacao_filename_prontuario = criar_relacao_filename_prontuario()
print(relacao_filename_prontuario)


In [ ]:
# Etapa 5: Função para organizar arquivos por prontuário dentro das pastas Dist Cond AtrioVent

def organizar_arquivos_por_prontuario(base_path, relacao_filename_prontuario):
    # Iterar sobre os diretórios dentro da pasta base (que são as pastas de Dist Cond AtrioVent)
    for dist_cond_folder in os.listdir(base_path):
        dist_cond_path = os.path.join(base_path, dist_cond_folder)
        if os.path.isdir(dist_cond_path):
            # Iterar sobre os arquivos dentro da subpasta de Dist Cond AtrioVent
            for arquivo in os.listdir(dist_cond_path):
                file_path = os.path.join(dist_cond_path, arquivo)
                if os.path.isfile(file_path):
                    filename_sem_extensao, _ = os.path.splitext(arquivo)
                    if filename_sem_extensao in relacao_filename_prontuario:
                        prontuario = str(relacao_filename_prontuario[filename_sem_extensao])
                        prontuario_folder = os.path.join(dist_cond_path, prontuario)
                        if not os.path.exists(prontuario_folder):
                            os.makedirs(prontuario_folder)
                        shutil.move(file_path, os.path.join(prontuario_folder, arquivo))
                    else:
                        print(f"Prontuário não encontrado para o arquivo {arquivo}")

# Chamar a função para organizar os arquivos por prontuário
organizar_arquivos_por_prontuario(base_path, relacao_filename_prontuario)


In [ ]:
# Etapa 6: Verificar a estrutura de diretórios e arquivos organizados

# Verificar a estrutura de diretórios e arquivos organizados
for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")
